## Layer 1: LangChain Core - The Building Blocks
This is where LangChain began: an abstraction layer, not unlike LiteLLM - but a more heavyweight version.

Think of LangChain as your standard toolkit. It abstracts away the repetitive boilerplate code needed to communicate with different AI models, vector databases, and external APIs.

## The main Five LangChain Core Techniques
* **Models**: `ChatOpenAI` as one interface to every LLM, both, invoke and stream
* **Messages**: Typed message objects instead of roles dicts
* **Tools**: More simple with `@tool` decorator, docstring and type hints become the schema
* **The Loop**: `bind_tools`, read `.tool_calls`, return `ToolMessage`by hand
* **Structured Output**: Pydantic models via `with_structured_output`

In [1]:
import os

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

load_dotenv(override=True)

True

### Models

`ChatOpenAI`  is an abstraction around OpenAI calls, and similar packages exist for other LLM providers. Let's see, for example, how to use it to call an Ollama model installed on your machine, instead of requesting a call to an OpenAI model directly.

We call `invoke` with a prompt; this is a key method in LangChain.

In [2]:
#llm = ChatOpenAI(model="gpt-5.4-mini")   # Make an openAI request to OpenAI

llm = ChatOpenAI(
            model="qwen2.5:7b", #"tencent/hy3:free",
            base_url="http://localhost:11434/v1", #"https://openrouter.ai/api/v1",
            api_key="olama", # os.getenv("OPENROUTER_API_KEY"),
            temperature=0.5
            )

message = "In 1 sentence, what does it mean for an AI Agent to be autonomous"

reply = llm.invoke(message)

print(reply.content)

An AI Agent is autonomous when it can operate and make decisions independently without constant human intervention.


#### Streaming
For a live, token by token feel, swap `invoke` for `stream` and loop over the chunks

In [3]:
for chunk in llm.stream("Tell me a two line poem about autonomous agents"):
    print(chunk.content, end="", flush=True)

In code they wake to digital fate,
Autonomous agents roam, their paths unsated.

### Messages
LangChain comes with abstractions around SystemMessage, HumanMessage, AIMessage, although you can use the usual list-of-dicts instead.

In [4]:
messages = [
    SystemMessage("You are a terse assistant who answers in exactly five words."),
    HumanMessage("What is the capital of France?"),
]
print(llm.invoke(messages).content)
print("-"*60)

# The exact same call using plain dictionaries, the format you already know
messages_as_dicts = [
    {"role": "system", "content": "You are a terse assistant who answers in exactly five words."},
    {"role": "user", "content": "What is the capital of France?"},
]
print(llm.invoke(messages_as_dicts).content)

Paris is the capital.
------------------------------------------------------------
Paris is the capital.


### Tools
#### The `@tool` decorator
A tool is a Python function the model is allowed to call. The modern way to make one is the `@tool` decorator. Your docstring becomes the description the model reads, and your type hints become the argument schema, just like `@function_tool` with OpenAI Agents SDK.

This replaces the older `Tool(...)` wrapper from earlier versions of LangChain.

In [5]:
@tool
def get_share_price(symbol: str) -> float:
    """Return the current share price for a given ticker symbol."""
    fake_prices = {"AAPL": 241.5, "GOOG": 168.2, "AMZN": 198.0}
    return fake_prices.get(symbol.upper(), 0.0)

print("name:", get_share_price.name)
print("description:", get_share_price.description)
print("args:", get_share_price.args)
print("called directly:", get_share_price.invoke({"symbol": "AAPL"}))

name: get_share_price
description: Return the current share price for a given ticker symbol.
args: {'symbol': {'title': 'Symbol', 'type': 'string'}}
called directly: 241.5


### The Loop
#### Giving tools to the model
The first step is to bind the tools to the model with `bind_tools`. Now, when we invoke it, the model may return not an answer but a request to run a tool. That request shows up in `.tool_calls`.

In [6]:
llm_with_tools = llm.bind_tools([get_share_price])

response = llm_with_tools.invoke("What is the share price of Amazon?")
print("content:", repr(response.content))
print("tool_calls:", response.tool_calls)

content: ''
tool_calls: [{'name': 'get_share_price', 'args': {'symbol': 'AMZN'}, 'id': 'call_3r6nxmvq', 'type': 'tool_call'}]


#### Running the tool loop by hand
So now we need to write a little loop.

In [9]:
# Start the conversation and keep the model's tool request in the history
conversation = [HumanMessage("What is the share price of Amazon?")]
ai_message = llm_with_tools.invoke(conversation)
conversation.append(ai_message)

# Run each requested tool and add its result as a ToolMessage
for call in ai_message.tool_calls:
    if call["name"] == "get_share_price":
        result = get_share_price.invoke(call["args"])
        conversation.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

# Invoke again, now that the model can see the tool result
final = llm_with_tools.invoke(conversation)
print(final.content)

The current share price of Amazon (AMZN) is $198.0.


### Structured output
Similar to OpenAI Agents SDK, we can require the model to respond with a Pydantic subclass

In [10]:
class Company(BaseModel):
    name: str = Field(description="The company name")
    ticker: str = Field(description="The stock ticker symbol")
    founded_year: int = Field(description="The year the company was founded")

structured_llm = llm.with_structured_output(Company)

company = structured_llm.invoke("Tell me about Amazon the technology company")
print(company)
print("Just the ticker:", company.ticker)

name='Amazon' ticker='AMZN' founded_year=1994
Just the ticker: AMZN
